# Linear — 45s motion films

Renders a film on a Colab instance and commits the finished MP4 back to the repo.

Two films share this pipeline:

| `FILM` | what it is |
|---|---|
| `launch` | the product launch film: overview, planning, issue tracking, cycles, command palette, agents, insights, release |
| `reel` | the abstract craft reel: eight motion studies, no product UI |

**Why this works.** The pipeline is a pure-Python frame generator — each frame is a function
of its index, rasterised by `rsvg-convert`, then encoded by ffmpeg. No browser, no GPU, no
framework. That makes it embarrassingly parallel across CPU cores, and a Colab instance has
more of them than a laptop.

**The GPU is not used, on purpose.** Neither the geometry (Python) nor the rasteriser
(`rsvg-convert`) has a GPU path. Setting the runtime to GPU changes nothing; CPU cores are
what matter. Pick a runtime with the most cores available.

**Requirements.** Colab ships Python and ffmpeg but none of the text or SVG stack, so the first
cell installs `librsvg2-bin` and Pango introspection. Pango matters for correctness: it is the
same shaper librsvg uses internally, so glyph positions computed here agree with what actually
gets drawn.

In [ ]:
FILM = "launch"        # "launch" or "reel"
WIDTH = 1920           # 756 for a fast timing pass

!pkill -f rsvg-convert; !pkill -f render.py
!rm -rf linear-motion-reel
!git clone --depth 1 https://github.com/emilio-n8n/linear-motion-reel.git
%cd linear-motion-reel
!bash tools/setup_env.sh

## Render

`JOBS` is capped just below the core count: `rsvg-convert` is single-threaded per frame, so
each worker is an independent process and throughput scales with cores.

Frames are cached in `.build/frames/<film>/`, so if the cell is interrupted, re-running it
resumes from the first missing frame instead of starting over.

In [ ]:
import os, sys, subprocess, time

# The apt install above puts python3-gi in Debian's dist-packages tree, which this
# interpreter does not search. Harmless if it is already on the path.
p = "/usr/lib/python3/dist-packages"
if os.path.isdir(p) and p not in sys.path:
    sys.path.append(p)

CORES = os.cpu_count() or 4
JOBS = max(1, CORES)
TOTAL = 1350
print(f"{CORES} cores -> {JOBS} workers, film={FILM}, width={WIDTH}", flush=True)

# Streamed so progress appears as it happens and Ctrl-C is not swallowed by a
# full pipe buffer.
proc = subprocess.Popen(
    ["python3", "src/render.py", "--film", FILM, "--width", str(WIDTH),
     "--jobs", str(JOBS)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

start = time.time()
for line in proc.stdout:
    line = line.strip()
    if not line:
        continue
    print(line, flush=True)
    if "/1350" in line:
        try:
            done = int(line.split("/")[0].split()[-1])
            el = time.time() - start
            rate = done / el if el else 0
            eta = (TOTAL - done) / rate if rate else 0
            print(f"    -> {100.0 * done / TOTAL:5.1f}%  {rate:5.1f} fps  "
                  f"elapsed {el / 60:4.1f}m  eta {eta / 60:4.1f}m", flush=True)
        except (ValueError, IndexError):
            pass

proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"render failed: {proc.returncode}")


## Encode

CRF 17 at `-preset slow`. The grain layer is high-frequency noise, so a low bitrate shows it as
blocking. For a quick look, `--crf 24 --preset veryfast` is roughly four times faster and fine
for judging timing.

In [ ]:
subprocess.run(
    ["python3", "src/encode.py", "--film", FILM, "--crf", "17", "--preset", "slow",
     "--width", str(WIDTH)],
    check=True,
)

from IPython.display import Video, display

out = "out/linear-launch-45s.mp4" if FILM == "launch" else "out/linear-craft-45s.mp4"
display(Video(out, embed=True, width=960))

## Commit the master back

This is the hand-off: the MP4 is committed to `main`, so it can be pulled locally with
`git pull` and lands in `out/`.

Create the token once per runtime as a Colab secret: **Secrets → Add new secret**, name
`GH_TOKEN`, value = a GitHub token with `repo` scope.

In [ ]:
import os, subprocess
from datetime import datetime, timezone

TOKEN = os.environ.get("GH_TOKEN", "")
if not TOKEN:
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
    print("using GH_TOKEN from Colab secrets")

REPO = "https://github.com/emilio-n8n/linear-motion-reel.git"
OUT = "out/linear-launch-45s.mp4" if FILM == "launch" else "out/linear-craft-45s.mp4"
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

subprocess.run(["git", "config", "user.name", "colab-render"], check=True)
subprocess.run(["git", "config", "user.email", "colab-render@users.noreply.github.com"], check=True)
subprocess.run(["git", "remote", "set-url", "origin",
                f"https://x-access-token:{TOKEN}@{REPO}"], check=True)

# Only the MP4 is ever staged: the intermediate frames are gitignored, and
# GitHub rejects pushes over ~100 MB.
subprocess.run(["git", "add", OUT], check=True)
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout

if not status.strip():
    print("MP4 unchanged — nothing to commit")
else:
    size_mb = os.path.getsize(OUT) / 1e6
    subprocess.run(
        ["git", "commit", "-m", f"render: {FILM} master ({size_mb:.1f} MB) {stamp}"],
        check=True,
    )
    subprocess.run(["git", "push", "origin", "main"], check=True)
    print(f"pushed {size_mb:.1f} MB")

## Pull it locally

```bash
cd ~/code/personal/motion-design-test
git pull origin main
open out/linear-launch-45s.mp4
```

Or fetch just the file without touching the working tree:

```bash
git fetch origin main
git show origin/main:out/linear-launch-45s.mp4 > out/linear-launch-45s.mp4
```